In [2]:
import json

T = CartanType(['A', 4])
W = WeylGroup(T,prefix="s")
[s1, s2, s3, s4] = W.simple_reflections()
KLP = json.load( open( "kl_polys/a4.json" ) )

WCR = WeylCharacterRing(T, style="coroots", base_ring=QQ)
R = WeightRing(WCR)
# EXPI MODIFIED, THIS CODE ONLY WORKS IN TYPE A
f = [w.coerce_to_sl() for w in WCR.fundamental_weights()]

n = len(W.simple_reflections())
e = s1^2
w0 = W.long_element()
S.<q> = LaurentPolynomialRing(QQ)
KL = KazhdanLusztigPolynomial(W,q)
q = 11

def Iw(w):
    a = []
    for i in range(1, n+1):
        if w.has_right_descent(i):
            a.append(i)
    return a

def Iwc(w):
    a = []
    for i in range(1, n+1):
        if not w.has_right_descent(i):
            #a.append(W.simple_reflections()[i])
            a.append(i)
    return a

def expI(l):
    a = R.one()
    for i in l:
        a = a * R(f[i-1])
    return a

def Cpsub_on_wr_element(w, v):
    interv = W.bruhat_interval(e, w)
    s = 0
    for y in interv:
        a = v.weyl_group_action(y)
        s += int(KLP[str((y, w))]) * a
    return s

def Cpsup_on_wr_element(w, v):
    interv = W.bruhat_interval(e, w0*w)
    s = 0
    for y in interv:
        a = v.weyl_group_action(w0*y)
        s += int(KLP[str((y, w0*w))]) * a
    return s

def fsub(w):
    return Cpsub_on_wr_element(w, expI(Iwc(w)))

def qfsub(w):
    return Cpsub_on_wr_element(w, expI(Iwc(w)).scale(q))

def fsup(w):
    return Cpsup_on_wr_element(w, expI(Iw(w)))

invols = []
for w in W:
    if w*w == e:
        invols.append(w)

def Iweight(w):
    wgt = 0
    for a in Iw(w):
        wgt += f[a-1]
    return wgt

rho = Iweight(w0)

def weightpairing(l):
    a = WCR.dot_reduce(l - rho)
    if a[0] == 0:
        return 0
    else:
        return a[0]*WCR(a[1])

def woke_pairing(a, b):
    #takes two elements of R and gives you an element of WCR
    d = dict(R(a*b))
    r = 0
    for l in d:
        r += d[l]*weightpairing(l)
    return r

invols = []
for w in W:
    if w*w == e:
        invols.append(w)

d_sub = {}
d_sup = {}

for i in range(len(W)):
    print(i + 1, "out of", len(W), end='\r')
    d_sub[i] = fsub(W[i])
    d_sup[i] = fsup(W[i])

print("Setup done, f_w, f^w are defined.")

Setup done, f_w, f^w are defined.


In [3]:
def build_matrix():
    m = []
    for i in range(len(W)):
        print(i + 1, "out of", len(W), end='\r')
        r = []
        for j in range(len(W)):
            r.append(woke_pairing(d_sup[i], d_sub[j]))
        m.append(r)
    return m
bigmat = build_matrix()
#bigmat[i][j] = woke_pairing(d_sub[i], d_sup[j]) for all i, j
alls = list(range(len(W)))
pops = list(range(len(W)))
order = []
while len(pops) > 0:
    for a in pops:
        good = True
        for b in pops:
            if b != a and bigmat[b][a] != 0:
                good = False
        if good:
            order.append(a)
            pops.remove(a)

order.reverse()
print("Computed the matrix of pairings and the upper triangular order.")

Computed the matrix of pairings and the upper triangular order.


In [4]:
def lc(i):
    a = bigmat[i][i]
    d = dict(WCR(a))
    if len(d) != 1:
        print("LC ERROR")
    else:
        return list(d.values())[0]

counter = 0
d_dual = {}
for j in order:
    counter += 1
    print(counter, "out of", len(W), end='\r')
    d_dual[j] = d_sub[j]/lc(j)
    for i in order:
        if  i != j and woke_pairing(d_sup[i], d_dual[j]) != 0:
            d_dual[j] = d_dual[j] - woke_pairing(d_sup[i], d_dual[j])*d_sub[i]/lc(i)
print("Computed the dual basis as the dictionary d_dual.")

Computed the dual basis as the dictionary d_dual.


In [5]:
#OPTIONAL CELL: CHECKING THE DUAL BASIS IS VALID.
valid = True
for i in range(len(W)):
    print(i + 1, "out of", len(W), end='\r')
    for j in range(len(W)):
        a = woke_pairing(d_sup[i], d_dual[j])
        if a != 0:
            if i != j or a != 1:
                print("ISSUE AT", i, j)
                valid = False
if valid:
    print("Checked dual basis, all is good.")
else:
    print("ISSUES! Dual basis is NOT correct.")

Checked dual basis, all is good.


In [6]:
newkl = {s1*s1: R(0,0,0,0), s3: 2*R(0,-1,0,0) + 2*R(-1,1,-1,0) + R(-1,0,1,-1) + R(-1,0,0,1) + 2*R(1,0,-1,0) + R(1,-1,1,-1) + R(1,-1,0,1) + R(0,1,0,-1) + R(0,1,-1,1), s3*s2: 2*R(0,0,-1,0) + R(0,-1,1,-1) + R(0,-1,0,1) + R(-1,1,0,-1) + R(-1,1,-1,1) + R(1,0,0,-1) + R(1,0,-1,1), s3*s4: R(-1,0,0,0) + R(1,-1,0,0) + R(0,1,-1,0), s3*s2*s1: R(0,0,0,-1) + R(0,0,-1,1), s3*s4*s2: 2*R(-1,0,-1,0) + R(-1,-1,1,-1) + R(-1,-1,0,1) + R(-2,1,0,-1) + R(-2,1,-1,1) + 2*R(1,-1,-1,0) + R(1,-2,1,-1) + R(1,-2,0,1) + 2*R(0,1,-2,0) + 5*R(0,0,0,-1) + 5*R(0,0,-1,1) + 2*R(0,-1,1,0) + R(-1,2,-1,-1) + R(-1,2,-2,1) + 2*R(-1,1,0,0) + R(2,-1,0,-1) + R(2,-1,-1,1) + R(1,1,-1,-1) + R(1,1,-2,1) + 2*R(1,0,0,0), s3*s4*s2*s1: 2*R(0,0,0,0) + R(-1,0,-1,1) + R(-1,0,0,-1) + R(1,-1,-1,1) + R(1,-1,0,-1) + R(0,1,-2,1) + R(0,1,-1,-1), s3*s4*s2*s3: R(0,-1,0,0) + R(-1,1,-1,0) + R(1,0,-1,0), s3*s4*s2*s3*s1: R(0,-1,0,-1) + R(0,-1,-1,1) + R(-1,1,-1,-1) + R(-1,1,-2,1) + 2*R(-1,0,0,0) + R(1,0,-1,-1) + R(1,0,-2,1) + 2*R(1,-1,0,0) + 2*R(0,1,-1,0), s3*s4*s2*s3*s1*s2: R(0,0,-1,0), s2: 2*R(0,0,-1,0) + 2*R(0,-1,1,-1) + 2*R(0,-1,0,1) + R(-1,1,0,-1) + R(-1,1,-1,1) + R(-1,0,1,0) + R(1,0,0,-1) + R(1,0,-1,1) + R(1,-1,1,0), s2*s3: 2*R(0,-1,0,0) + R(-1,1,-1,0) + R(-1,0,1,-1) + R(-1,0,0,1) + R(1,0,-1,0) + R(1,-1,1,-1) + R(1,-1,0,1), s2*s3*s2: 16*R(0,0,0,0) + 2*R(0,-2,0,1) + 2*R(0,-2,1,-1) + 4*R(0,-1,-1,0) + 3*R(-2,1,0,0) + R(-2,2,-2,1) + R(-2,2,-1,-1) + R(-2,1,1,-2) + R(-2,1,-1,2) + 2*R(-1,1,-2,0) + 5*R(-1,-1,1,0) + R(-1,-1,2,-2) + R(-1,-1,0,2) + 8*R(-1,0,-1,1) + 8*R(-1,0,0,-1) + 3*R(2,-1,0,0) + R(2,0,-2,1) + R(2,0,-1,-1) + R(2,-1,1,-2) + R(2,-1,-1,2) + 2*R(1,1,-1,0) + R(1,0,1,-1) + R(1,0,0,1) + 2*R(1,0,-2,0) + 5*R(1,-2,1,0) + R(1,-2,2,-2) + R(1,-2,0,2) + 8*R(1,-1,-1,1) + 8*R(1,-1,0,-1) + 2*R(-1,2,-1,0) + R(-1,1,1,-1) + R(-1,1,0,1) + 5*R(0,1,-2,1) + 5*R(0,1,-1,-1) + 2*R(0,-1,2,-1) + 2*R(0,-1,1,1) + 3*R(0,0,1,-2) + 3*R(0,0,-1,2), s2*s3*s4: R(-1,0,0,0) + R(1,-1,0,0), s2*s3*s2*s1: 2*R(0,-1,0,-1) + 2*R(0,-1,-1,1) + R(-1,1,-1,-1) + R(-1,1,-2,1) + R(-1,0,1,-2) + 4*R(-1,0,0,0) + R(-1,0,-1,2) + R(1,0,-1,-1) + R(1,0,-2,1) + R(1,-1,1,-2) + 4*R(1,-1,0,0) + R(1,-1,-1,2) + 2*R(0,1,-1,0) + R(0,0,1,-1) + R(0,0,0,1), s2*s3*s4*s2: 2*R(-1,0,-1,0) + R(-1,-1,1,-1) + R(-1,-1,0,1) + R(-2,1,0,-1) + R(-2,1,-1,1) + 2*R(1,-1,-1,0) + R(1,-2,1,-1) + R(1,-2,0,1) + 4*R(0,0,0,-1) + 4*R(0,0,-1,1) + 2*R(0,-1,1,0) + R(-1,1,0,0) + R(2,-1,0,-1) + R(2,-1,-1,1) + R(1,0,0,0), s2*s3*s4*s2*s1: 2*R(0,0,0,0) + R(-1,0,-1,1) + R(-1,0,0,-1) + R(1,-1,-1,1) + R(1,-1,0,-1), s2*s3*s4*s2*s3: R(-1,-1,0,0) + R(-2,1,-1,0) + R(1,-2,0,0) + 4*R(0,0,-1,0) + 2*R(0,-1,1,-1) + 2*R(0,-1,0,1) + R(-1,1,0,-1) + R(-1,1,-1,1) + R(2,-1,-1,0) + R(1,0,0,-1) + R(1,0,-1,1), s2*s3*s4*s2*s3*s1: R(-1,-1,0,-1) + R(-1,-1,-1,1) + R(-2,1,-1,-1) + R(-2,1,-2,1) + 2*R(-2,0,0,0) + R(1,-2,0,-1) + R(1,-2,-1,1) + 3*R(0,0,-1,-1) + 3*R(0,0,-2,1) + 2*R(0,-1,1,-2) + 9*R(0,-1,0,0) + 2*R(0,-1,-1,2) + R(-1,1,0,-2) + 6*R(-1,1,-1,0) + R(-1,1,-2,2) + 3*R(-1,0,1,-1) + 3*R(-1,0,0,1) + R(2,-1,-1,-1) + R(2,-1,-2,1) + 2*R(2,-2,0,0) + R(1,0,0,-2) + 6*R(1,0,-1,0) + R(1,0,-2,2) + 3*R(1,-1,1,-1) + 3*R(1,-1,0,1) + 2*R(0,1,0,-1) + 2*R(0,1,-1,1), s2*s3*s4*s2*s3*s1*s2: R(-1,0,-1,0) + R(1,-1,-1,0) + R(0,0,0,-1) + R(0,0,-1,1), s2*s1: R(0,0,0,-1) + R(0,0,-1,1) + R(0,-1,1,0), s2*s3*s1: 2*R(0,-1,0,-1) + 2*R(0,-1,-1,1) + 2*R(0,-2,1,0) + R(-1,1,-1,-1) + R(-1,1,-2,1) + R(-1,0,1,-2) + 5*R(-1,0,0,0) + R(-1,0,-1,2) + R(-1,-1,2,-1) + R(-1,-1,1,1) + R(1,0,-1,-1) + R(1,0,-2,1) + R(1,-1,1,-2) + 5*R(1,-1,0,0) + R(1,-1,-1,2) + R(1,-2,2,-1) + R(1,-2,1,1) + 2*R(0,1,-1,0) + 2*R(0,0,1,-1) + 2*R(0,0,0,1), s2*s3*s1*s2: R(0,0,-1,0) + R(0,-1,1,-1) + R(0,-1,0,1), s2*s3*s4*s1: 2*R(0,0,0,0) + R(-1,-1,1,0) + R(-1,0,-1,1) + R(-1,0,0,-1) + R(1,-2,1,0) + R(1,-1,-1,1) + R(1,-1,0,-1), s2*s3*s1*s2*s1: R(0,0,-1,-1) + R(0,0,-2,1) + R(0,-1,1,-2) + 4*R(0,-1,0,0) + R(0,-1,-1,2) + 2*R(-1,1,-1,0) + R(-1,0,1,-1) + R(-1,0,0,1) + 2*R(1,0,-1,0) + R(1,-1,1,-1) + R(1,-1,0,1), s2*s3*s4*s1*s2: R(-1,0,-1,0) + R(-1,-1,1,-1) + R(-1,-1,0,1) + R(1,-1,-1,0) + R(1,-2,1,-1) + R(1,-2,0,1) + 2*R(0,0,0,-1) + 2*R(0,0,-1,1) + 2*R(0,-1,1,0), s2*s3*s4*s1*s2*s1: R(-1,0,-1,-1) + R(-1,0,-2,1) + R(-1,-1,1,-2) + 3*R(-1,-1,0,0) + R(-1,-1,-1,2) + 2*R(-2,1,-1,0) + R(-2,0,1,-1) + R(-2,0,0,1) + R(1,-1,-1,-1) + R(1,-1,-2,1) + R(1,-2,1,-2) + 3*R(1,-2,0,0) + R(1,-2,-1,2) + 2*R(0,0,0,-2) + 9*R(0,0,-1,0) + 2*R(0,0,-2,2) + 6*R(0,-1,1,-1) + 6*R(0,-1,0,1) + 3*R(-1,1,0,-1) + 3*R(-1,1,-1,1) + 2*R(-1,0,1,0) + 2*R(2,-1,-1,0) + R(2,-2,1,-1) + R(2,-2,0,1) + 3*R(1,0,0,-1) + 3*R(1,0,-1,1) + 2*R(1,-1,1,0), s2*s3*s4*s1*s2*s3: R(0,-1,0,0), s2*s3*s4*s1*s2*s3*s1: R(0,-1,0,-1) + R(0,-1,-1,1) + R(-1,0,0,0) + R(1,-1,0,0), s2*s3*s4*s1*s2*s3*s1*s2: R(0,0,0,0) + R(0,-1,-1,0) + R(-1,0,-1,1) + R(-1,0,0,-1) + R(1,-1,-1,1) + R(1,-1,0,-1), s4: R(-1,0,0,0) + R(1,-1,0,0) + R(0,1,-1,0) + R(0,0,1,-1), s4*s3: R(0,-1,0,0) + R(-1,1,-1,0) + R(-1,0,1,-1) + R(1,0,-1,0) + R(1,-1,1,-1) + R(0,1,0,-1), s4*s3*s2: R(0,0,-1,0) + R(0,-1,1,-1) + R(-1,1,0,-1) + R(1,0,0,-1), s3*s4*s3: R(-1,-1,0,0) + R(-2,1,-1,0) + R(-2,0,1,-1) + R(1,-2,0,0) + 5*R(0,0,-1,0) + 3*R(0,-1,1,-1) + 2*R(0,-1,0,1) + R(-1,2,-2,0) + 3*R(-1,1,0,-1) + 2*R(-1,1,-1,1) + R(-1,0,1,0) + R(2,-1,-1,0) + R(2,-2,1,-1) + R(1,1,-2,0) + 3*R(1,0,0,-1) + 2*R(1,0,-1,1) + R(1,-1,1,0) + R(0,2,-1,-1) + R(0,1,0,0), s4*s3*s2*s1: R(0,0,0,-1), s3*s4*s3*s2: R(-1,0,-1,0) + R(-1,-1,1,-1) + R(-2,1,0,-1) + R(1,-1,-1,0) + R(1,-2,1,-1) + R(0,1,-2,0) + 4*R(0,0,0,-1) + 2*R(0,0,-1,1) + R(0,-1,1,0) + R(-1,2,-1,-1) + R(-1,1,0,0) + R(2,-1,0,-1) + R(1,1,-1,-1) + R(1,0,0,0), s3*s4*s3*s2*s1: R(0,0,0,0) + R(-1,0,0,-1) + R(1,-1,0,-1) + R(0,1,-1,-1), s3*s4*s2*s3*s2: 5*R(0,0,0,0) + R(0,-2,1,-1) + R(0,-1,-1,0) + R(-2,1,0,0) + R(-2,2,-1,-1) + R(-1,1,-2,0) + R(-1,-1,1,0) + 2*R(-1,0,-1,1) + 3*R(-1,0,0,-1) + R(2,-1,0,0) + R(2,0,-1,-1) + R(1,1,-1,0) + R(1,0,-2,0) + R(1,-2,1,0) + 2*R(1,-1,-1,1) + 3*R(1,-1,0,-1) + R(-1,2,-1,0) + 2*R(0,1,-2,1) + 3*R(0,1,-1,-1), s3*s4*s2*s3*s2*s1: R(0,-1,0,-1) + R(-1,1,-1,-1) + R(-1,0,0,0) + R(1,0,-1,-1) + R(1,-1,0,0) + R(0,1,-1,0), s3*s4*s2*s3*s1*s2*s1: R(0,0,-1,-1) + R(0,-1,0,0) + R(-1,1,-1,0) + R(1,0,-1,0), s4*s2: 3*R(-1,0,-1,0) + 3*R(-1,-1,1,-1) + 2*R(-1,-1,0,1) + 2*R(-2,1,0,-1) + R(-2,1,-1,1) + R(-2,0,1,0) + 3*R(1,-1,-1,0) + 3*R(1,-2,1,-1) + 2*R(1,-2,0,1) + 2*R(0,1,-2,0) + 9*R(0,0,0,-1) + 6*R(0,0,-1,1) + 2*R(0,-1,2,-2) + 6*R(0,-1,1,0) + R(-1,2,-1,-1) + R(-1,2,-2,1) + R(-1,1,1,-2) + 3*R(-1,1,0,0) + R(-1,0,2,-1) + 2*R(2,-1,0,-1) + R(2,-1,-1,1) + R(2,-2,1,0) + R(1,1,-1,-1) + R(1,1,-2,1) + R(1,0,1,-2) + 3*R(1,0,0,0) + R(1,-1,2,-1), s4*s2*s3: 2*R(0,-1,0,0) + R(-1,1,-1,0) + R(-1,0,1,-1) + R(1,0,-1,0) + R(1,-1,1,-1), s4*s2*s3*s2: 9*R(0,0,0,0) + 2*R(0,-2,1,-1) + 2*R(0,-1,-1,0) + 2*R(-2,1,0,0) + R(-2,2,-1,-1) + R(-2,1,1,-2) + R(-1,1,-2,0) + 3*R(-1,-1,1,0) + R(-1,-1,2,-2) + 3*R(-1,0,-1,1) + 6*R(-1,0,0,-1) + 2*R(2,-1,0,0) + R(2,0,-1,-1) + R(2,-1,1,-2) + R(1,1,-1,0) + R(1,0,1,-1) + R(1,0,-2,0) + 3*R(1,-2,1,0) + R(1,-2,2,-2) + 3*R(1,-1,-1,1) + 6*R(1,-1,0,-1) + R(-1,2,-1,0) + R(-1,1,1,-1) + 2*R(0,1,-2,1) + 3*R(0,1,-1,-1) + 2*R(0,-1,2,-1) + 3*R(0,0,1,-2), s2*s3*s4*s3: R(-1,-1,0,0) + R(-2,1,-1,0) + R(-2,0,1,-1) + R(1,-2,0,0) + 4*R(0,0,-1,0) + 4*R(0,-1,1,-1) + 2*R(0,-1,0,1) + 2*R(-1,1,0,-1) + R(-1,1,-1,1) + R(-1,0,1,0) + R(2,-1,-1,0) + R(2,-2,1,-1) + 2*R(1,0,0,-1) + R(1,0,-1,1) + R(1,-1,1,0), s4*s2*s3*s2*s1: 2*R(0,-1,0,-1) + R(-1,1,-1,-1) + R(-1,0,1,-2) + 2*R(-1,0,0,0) + R(1,0,-1,-1) + R(1,-1,1,-2) + 2*R(1,-1,0,0) + R(0,1,-1,0) + R(0,0,1,-1), s2*s3*s4*s3*s2: R(-1,0,-1,0) + R(-1,-1,1,-1) + R(-2,1,0,-1) + R(1,-1,-1,0) + R(1,-2,1,-1) + 4*R(0,0,0,-1) + 2*R(0,0,-1,1) + 2*R(0,-1,1,0) + R(-1,1,0,0) + R(2,-1,0,-1) + R(1,0,0,0), s2*s3*s4*s3*s2*s1: R(0,0,0,0) + R(-1,0,0,-1) + R(1,-1,0,-1), s2*s3*s4*s2*s3*s2: R(-1,-1,-1,0) + R(-1,-2,1,-1) + R(-2,1,-2,0) + 3*R(-2,0,0,-1) + 2*R(-2,0,-1,1) + R(-2,-1,1,0) + R(-3,2,-1,-1) + R(-3,1,0,0) + R(1,-2,-1,0) + R(1,-3,1,-1) + 3*R(0,0,-2,0) + 13*R(0,-1,0,-1) + 9*R(0,-1,-1,1) + 2*R(0,-2,2,-2) + 6*R(0,-2,1,0) + 9*R(-1,1,-1,-1) + 6*R(-1,1,-2,1) + 6*R(-1,0,1,-2) + 21*R(-1,0,0,0) + 3*R(-1,0,-1,2) + 3*R(-1,-1,2,-1) + 2*R(-1,-1,1,1) + R(-2,2,0,-2) + 3*R(-2,2,-1,0) + 2*R(-2,1,1,-1) + R(-2,1,0,1) + R(2,-1,-2,0) + 3*R(2,-2,0,-1) + 2*R(2,-2,-1,1) + R(2,-3,1,0) + 9*R(1,0,-1,-1) + 6*R(1,0,-2,1) + 6*R(1,-1,1,-2) + 21*R(1,-1,0,0) + 3*R(1,-1,-1,2) + 3*R(1,-2,2,-1) + 2*R(1,-2,1,1) + 3*R(0,1,0,-2) + 13*R(0,1,-1,0) + 2*R(0,1,-2,2) + 9*R(0,0,1,-1) + 6*R(0,0,0,1) + R(-1,2,0,-1) + R(-1,2,-1,1) + R(3,-1,-1,-1) + R(3,-2,0,0) + R(2,0,0,-2) + 3*R(2,0,-1,0) + 2*R(2,-1,1,-1) + R(2,-1,0,1) + R(1,1,0,-1) + R(1,1,-1,1), s2*s3*s4*s2*s3*s2*s1: R(-1,-1,0,-1) + R(-2,1,-1,-1) + R(-2,0,0,0) + R(1,-2,0,-1) + 3*R(0,0,-1,-1) + 2*R(0,-1,1,-2) + 5*R(0,-1,0,0) + R(-1,1,0,-2) + 3*R(-1,1,-1,0) + 2*R(-1,0,1,-1) + R(-1,0,0,1) + R(2,-1,-1,-1) + R(2,-2,0,0) + R(1,0,0,-2) + 3*R(1,0,-1,0) + 2*R(1,-1,1,-1) + R(1,-1,0,1) + R(0,1,0,-1) + R(0,1,-1,1), s2*s3*s4*s2*s3*s1*s2*s1: R(-1,0,-1,-1) + R(-1,-1,0,0) + R(-2,1,-1,0) + R(1,-1,-1,-1) + R(1,-2,0,0) + R(0,0,0,-2) + 4*R(0,0,-1,0) + 2*R(0,-1,1,-1) + R(0,-1,0,1) + R(-1,1,0,-1) + R(-1,1,-1,1) + R(2,-1,-1,0) + R(1,0,0,-1) + R(1,0,-1,1), s4*s2*s1: 4*R(0,0,0,0) + R(-1,-1,1,0) + R(-1,0,-1,1) + 2*R(-1,0,0,-1) + R(1,-2,1,0) + R(1,-1,-1,1) + 2*R(1,-1,0,-1) + R(0,1,-2,1) + R(0,1,-1,-1) + R(0,-1,2,-1) + R(0,0,1,-2), s4*s2*s3*s1: 2*R(0,-1,0,-1) + R(0,-1,-1,1) + R(0,-2,1,0) + R(-1,1,-1,-1) + R(-1,1,-2,1) + R(-1,0,1,-2) + 4*R(-1,0,0,0) + R(-1,-1,2,-1) + R(1,0,-1,-1) + R(1,0,-2,1) + R(1,-1,1,-2) + 4*R(1,-1,0,0) + R(1,-2,2,-1) + 2*R(0,1,-1,0) + 2*R(0,0,1,-1), s4*s2*s3*s1*s2: R(0,0,-1,0) + R(0,-1,1,-1), s2*s3*s4*s3*s1: 2*R(-1,-1,0,-1) + R(-1,-1,-1,1) + R(-1,-2,1,0) + R(-2,1,-1,-1) + R(-2,1,-2,1) + R(-2,0,1,-2) + 3*R(-2,0,0,0) + R(-2,-1,2,-1) + 2*R(1,-2,0,-1) + R(1,-2,-1,1) + R(1,-3,1,0) + 5*R(0,0,-1,-1) + 3*R(0,0,-2,1) + 5*R(0,-1,1,-2) + 16*R(0,-1,0,0) + 2*R(0,-1,-1,2) + 3*R(0,-2,2,-1) + 2*R(0,-2,1,1) + 2*R(-1,1,0,-2) + 8*R(-1,1,-1,0) + R(-1,1,-2,2) + 8*R(-1,0,1,-1) + 5*R(-1,0,0,1) + R(-1,-1,2,0) + R(2,-1,-1,-1) + R(2,-1,-2,1) + R(2,-2,1,-2) + 3*R(2,-2,0,0) + R(2,-3,2,-1) + 2*R(1,0,0,-2) + 8*R(1,0,-1,0) + R(1,0,-2,2) + 8*R(1,-1,1,-1) + 5*R(1,-1,0,1) + R(1,-2,2,0) + 4*R(0,1,0,-1) + 2*R(0,1,-1,1) + 2*R(0,0,1,0), s4*s2*s3*s1*s2*s1: R(0,0,-1,-1) + R(0,-1,1,-2) + 2*R(0,-1,0,0) + R(-1,1,-1,0) + R(-1,0,1,-1) + R(1,0,-1,0) + R(1,-1,1,-1), s2*s3*s4*s3*s1*s2: R(-1,0,-1,0) + R(-1,-1,1,-1) + R(1,-1,-1,0) + R(1,-2,1,-1) + 2*R(0,0,0,-1) + R(0,0,-1,1) + R(0,-1,1,0), s2*s3*s4*s3*s1*s2*s1: R(-1,0,-1,-1) + R(-1,-1,1,-2) + 2*R(-1,-1,0,0) + R(-2,1,-1,0) + R(-2,0,1,-1) + R(1,-1,-1,-1) + R(1,-2,1,-2) + 2*R(1,-2,0,0) + 2*R(0,0,0,-2) + 5*R(0,0,-1,0) + 5*R(0,-1,1,-1) + 2*R(0,-1,0,1) + 2*R(-1,1,0,-1) + R(-1,1,-1,1) + R(-1,0,1,0) + R(2,-1,-1,0) + R(2,-2,1,-1) + 2*R(1,0,0,-1) + R(1,0,-1,1) + R(1,-1,1,0), s2*s3*s4*s1*s2*s3*s2: 2*R(0,0,0,0) + R(0,-2,1,-1) + R(0,-1,-1,0) + R(-1,-1,1,0) + R(-1,0,-1,1) + 2*R(-1,0,0,-1) + R(1,-2,1,0) + R(1,-1,-1,1) + 2*R(1,-1,0,-1), s2*s3*s4*s1*s2*s3*s2*s1: R(0,-1,0,-1) + R(-1,0,0,0) + R(1,-1,0,0), s2*s3*s4*s1*s2*s3*s1*s2*s1: R(0,-1,-1,-1) + R(0,-2,0,0) + R(-1,0,0,-2) + 3*R(-1,0,-1,0) + 2*R(-1,-1,1,-1) + R(-1,-1,0,1) + R(-2,1,0,-1) + R(-2,1,-1,1) + R(1,-1,0,-2) + 3*R(1,-1,-1,0) + 2*R(1,-2,1,-1) + R(1,-2,0,1) + 5*R(0,0,0,-1) + 3*R(0,0,-1,1) + 2*R(0,-1,1,0) + R(-1,1,0,0) + R(2,-1,0,-1) + R(2,-1,-1,1) + R(1,0,0,0), s1: R(0,0,0,-1) + R(0,0,-1,1) + R(0,-1,1,0) + R(-1,1,0,0), s3*s1: 3*R(0,-1,0,-1) + 3*R(0,-1,-1,1) + 2*R(0,-2,1,0) + 3*R(-1,1,-1,-1) + 3*R(-1,1,-2,1) + 2*R(-1,0,1,-2) + 9*R(-1,0,0,0) + 2*R(-1,0,-1,2) + R(-1,-1,2,-1) + R(-1,-1,1,1) + 2*R(-2,2,-1,0) + R(-2,1,1,-1) + R(-2,1,0,1) + 2*R(1,0,-1,-1) + 2*R(1,0,-2,1) + R(1,-1,1,-2) + 6*R(1,-1,0,0) + R(1,-1,-1,2) + R(1,-2,2,-1) + R(1,-2,1,1) + R(0,1,0,-2) + 6*R(0,1,-1,0) + R(0,1,-2,2) + 3*R(0,0,1,-1) + 3*R(0,0,0,1) + R(-1,2,0,-1) + R(-1,2,-1,1), s3*s1*s2: 2*R(0,0,-1,0) + R(0,-1,1,-1) + R(0,-1,0,1) + R(-1,1,0,-1) + R(-1,1,-1,1), s3*s4*s1: 4*R(0,0,0,0) + R(-2,1,0,0) + R(-1,-1,1,0) + 2*R(-1,0,-1,1) + 2*R(-1,0,0,-1) + R(1,-2,1,0) + R(1,-1,-1,1) + R(1,-1,0,-1) + R(-1,2,-1,0) + R(0,1,-2,1) + R(0,1,-1,-1), s3*s1*s2*s1: R(0,0,-1,-1) + R(0,0,-2,1) + R(0,-1,1,-2) + 4*R(0,-1,0,0) + R(0,-1,-1,2) + R(-1,1,0,-2) + 4*R(-1,1,-1,0) + R(-1,1,-2,2) + 2*R(-1,0,1,-1) + 2*R(-1,0,0,1) + 2*R(1,0,-1,0) + R(1,-1,1,-1) + R(1,-1,0,1) + R(0,1,0,-1) + R(0,1,-1,1), s3*s4*s1*s2: 2*R(-1,0,-1,0) + R(-1,-1,1,-1) + R(-1,-1,0,1) + R(-2,1,0,-1) + R(-2,1,-1,1) + R(1,-1,-1,0) + R(1,-2,1,-1) + R(1,-2,0,1) + R(0,1,-2,0) + 4*R(0,0,0,-1) + 4*R(0,0,-1,1) + 2*R(0,-1,1,0) + R(-1,2,-1,-1) + R(-1,2,-2,1) + 2*R(-1,1,0,0), s3*s4*s1*s2*s1: 2*R(-1,0,-1,-1) + 2*R(-1,0,-2,1) + R(-1,-1,1,-2) + 5*R(-1,-1,0,0) + R(-1,-1,-1,2) + R(-2,1,0,-2) + 5*R(-2,1,-1,0) + R(-2,1,-2,2) + 2*R(-2,0,1,-1) + 2*R(-2,0,0,1) + R(1,-1,-1,-1) + R(1,-1,-2,1) + R(1,-2,1,-2) + 3*R(1,-2,0,0) + R(1,-2,-1,2) + R(0,1,-2,-1) + R(0,1,-3,1) + 3*R(0,0,0,-2) + 16*R(0,0,-1,0) + 3*R(0,0,-2,2) + 8*R(0,-1,1,-1) + 8*R(0,-1,0,1) + R(-1,2,-1,-2) + 3*R(-1,2,-2,0) + R(-1,2,-3,2) + 8*R(-1,1,0,-1) + 8*R(-1,1,-1,1) + 4*R(-1,0,1,0) + 2*R(2,-1,-1,0) + R(2,-2,1,-1) + R(2,-2,0,1) + 2*R(1,1,-2,0) + 5*R(1,0,0,-1) + 5*R(1,0,-1,1) + 2*R(1,-1,1,0) + R(0,2,-1,-1) + R(0,2,-2,1) + 2*R(0,1,0,0), s3*s4*s1*s2*s3: R(0,-1,0,0) + R(-1,1,-1,0), s3*s4*s1*s2*s3*s1: R(0,-1,0,-1) + R(0,-1,-1,1) + R(-1,1,-1,-1) + R(-1,1,-2,1) + 2*R(-1,0,0,0) + R(1,-1,0,0) + R(0,1,-1,0), s3*s4*s1*s2*s3*s1*s2: 2*R(0,0,0,0) + R(0,-1,-1,0) + R(-1,1,-2,0) + 2*R(-1,0,-1,1) + 2*R(-1,0,0,-1) + R(1,-1,-1,1) + R(1,-1,0,-1) + R(0,1,-2,1) + R(0,1,-1,-1), s1*s2: R(0,0,-1,0) + R(0,-1,1,-1) + R(0,-1,0,1) + R(-1,1,0,-1) + R(-1,1,-1,1) + R(-1,0,1,0), s1*s2*s3: R(0,-1,0,0) + R(-1,1,-1,0) + R(-1,0,1,-1) + R(-1,0,0,1), s1*s2*s3*s2: 9*R(0,0,0,0) + R(0,-2,0,1) + R(0,-2,1,-1) + 2*R(0,-1,-1,0) + 3*R(-2,1,0,0) + R(-2,2,-2,1) + R(-2,2,-1,-1) + R(-2,1,1,-2) + R(-2,1,-1,2) + 2*R(-1,1,-2,0) + 3*R(-1,-1,1,0) + R(-1,-1,2,-2) + R(-1,-1,0,2) + 6*R(-1,0,-1,1) + 6*R(-1,0,0,-1) + 2*R(1,-2,1,0) + 3*R(1,-1,-1,1) + 3*R(1,-1,0,-1) + 2*R(-1,2,-1,0) + R(-1,1,1,-1) + R(-1,1,0,1) + 3*R(0,1,-2,1) + 3*R(0,1,-1,-1) + R(0,-1,2,-1) + R(0,-1,1,1) + 2*R(0,0,1,-2) + 2*R(0,0,-1,2), s1*s2*s3*s4: R(-1,0,0,0), s1*s2*s3*s2*s1: R(0,-1,0,-1) + R(0,-1,-1,1) + R(-1,1,-1,-1) + R(-1,1,-2,1) + R(-1,0,1,-2) + 4*R(-1,0,0,0) + R(-1,0,-1,2) + 2*R(1,-1,0,0) + 2*R(0,1,-1,0) + R(0,0,1,-1) + R(0,0,0,1), s1*s2*s3*s4*s2: 2*R(-1,0,-1,0) + R(-1,-1,1,-1) + R(-1,-1,0,1) + R(-2,1,0,-1) + R(-2,1,-1,1) + 2*R(0,0,0,-1) + 2*R(0,0,-1,1) + R(0,-1,1,0) + R(-1,1,0,0), s1*s2*s3*s4*s2*s1: R(0,0,0,0) + R(-1,0,-1,1) + R(-1,0,0,-1), s1*s2*s3*s4*s2*s3: R(-1,-1,0,0) + R(-2,1,-1,0) + 2*R(0,0,-1,0) + R(0,-1,1,-1) + R(0,-1,0,1) + R(-1,1,0,-1) + R(-1,1,-1,1), s1*s2*s3*s4*s2*s3*s1: R(-1,-1,0,-1) + R(-1,-1,-1,1) + R(-2,1,-1,-1) + R(-2,1,-2,1) + 2*R(-2,0,0,0) + 2*R(0,0,-1,-1) + 2*R(0,0,-2,1) + R(0,-1,1,-2) + 5*R(0,-1,0,0) + R(0,-1,-1,2) + R(-1,1,0,-2) + 5*R(-1,1,-1,0) + R(-1,1,-2,2) + 2*R(-1,0,1,-1) + 2*R(-1,0,0,1) + 2*R(1,0,-1,0) + R(1,-1,1,-1) + R(1,-1,0,1) + R(0,1,0,-1) + R(0,1,-1,1), s1*s2*s3*s4*s2*s3*s1*s2: R(-1,0,-1,0) + R(0,0,0,-1) + R(0,0,-1,1), s1*s2*s1: R(0,0,-1,-1) + R(0,0,-2,1) + R(0,-1,1,-2) + 5*R(0,-1,0,0) + R(0,-1,-1,2) + R(0,-2,2,-1) + R(0,-2,1,1) + R(-1,1,0,-2) + 3*R(-1,1,-1,0) + R(-1,1,-2,2) + 3*R(-1,0,1,-1) + 3*R(-1,0,0,1) + R(-1,-1,2,0) + 2*R(1,0,-1,0) + 2*R(1,-1,1,-1) + 2*R(1,-1,0,1) + R(0,1,0,-1) + R(0,1,-1,1) + R(0,0,1,0), s1*s2*s3*s1: R(0,-1,0,-1) + R(0,-1,-1,1) + R(0,-2,1,0) + R(-1,1,-1,-1) + R(-1,1,-2,1) + R(-1,0,1,-2) + 4*R(-1,0,0,0) + R(-1,0,-1,2) + R(-1,-1,2,-1) + R(-1,-1,1,1) + 2*R(1,-1,0,0) + R(0,1,-1,0) + R(0,0,1,-1) + R(0,0,0,1), s1*s2*s3*s1*s2: 5*R(0,0,0,0) + R(0,-2,0,1) + R(0,-2,1,-1) + R(0,-1,-1,0) + R(-1,1,-2,0) + 3*R(-1,-1,1,0) + R(-1,-1,2,-2) + R(-1,-1,0,2) + 3*R(-1,0,-1,1) + 3*R(-1,0,0,-1) + 2*R(1,-2,1,0) + 2*R(1,-1,-1,1) + 2*R(1,-1,0,-1) + R(0,1,-2,1) + R(0,1,-1,-1) + R(0,-1,2,-1) + R(0,-1,1,1) + R(0,0,1,-2) + R(0,0,-1,2), s1*s2*s3*s4*s1: R(0,0,0,0) + R(-1,-1,1,0) + R(-1,0,-1,1) + R(-1,0,0,-1), s1*s2*s3*s1*s2*s1: R(0,-1,-1,-1) + R(0,-1,-2,1) + R(0,-2,1,-2) + 3*R(0,-2,0,0) + R(0,-2,-1,2) + R(-1,1,-2,-1) + R(-1,1,-3,1) + 3*R(-1,0,0,-2) + 13*R(-1,0,-1,0) + 3*R(-1,0,-2,2) + R(-1,-1,2,-3) + 9*R(-1,-1,1,-1) + 9*R(-1,-1,0,1) + R(-1,-1,-1,3) + 2*R(-2,2,-2,0) + 6*R(-2,1,0,-1) + 6*R(-2,1,-1,1) + R(-2,0,2,-2) + 3*R(-2,0,1,0) + R(-2,0,0,2) + 2*R(1,-1,0,-2) + 9*R(1,-1,-1,0) + 2*R(1,-1,-2,2) + 6*R(1,-2,1,-1) + 6*R(1,-2,0,1) + R(0,1,-1,-2) + 6*R(0,1,-2,0) + R(0,1,-3,2) + R(0,0,1,-3) + 21*R(0,0,0,-1) + 21*R(0,0,-1,1) + R(0,0,-2,3) + 3*R(0,-1,2,-2) + 13*R(0,-1,1,0) + 3*R(0,-1,0,2) + 3*R(-1,2,-1,-1) + 3*R(-1,2,-2,1) + 2*R(-1,1,1,-2) + 9*R(-1,1,0,0) + 2*R(-1,1,-1,2) + R(-1,0,2,-1) + R(-1,0,1,1) + 3*R(2,-1,0,-1) + 3*R(2,-1,-1,1) + 2*R(2,-2,1,0) + 2*R(1,1,-1,-1) + 2*R(1,1,-2,1) + R(1,0,1,-2) + 6*R(1,0,0,0) + R(1,0,-1,2) + R(1,-1,2,-1) + R(1,-1,1,1), s1*s2*s3*s4*s1*s2: R(-1,0,-1,0) + R(-1,-1,1,-1) + R(-1,-1,0,1) + R(0,0,0,-1) + R(0,0,-1,1) + R(0,-1,1,0), s1*s2*s3*s4*s1*s2*s1: R(-1,0,-1,-1) + R(-1,0,-2,1) + R(-1,-1,1,-2) + 3*R(-1,-1,0,0) + R(-1,-1,-1,2) + 2*R(-2,1,-1,0) + R(-2,0,1,-1) + R(-2,0,0,1) + R(0,0,0,-2) + 5*R(0,0,-1,0) + R(0,0,-2,2) + 3*R(0,-1,1,-1) + 3*R(0,-1,0,1) + 2*R(-1,1,0,-1) + 2*R(-1,1,-1,1) + R(-1,0,1,0) + R(1,0,0,-1) + R(1,0,-1,1) + R(1,-1,1,0), s1*s2*s3*s4*s1*s2*s3: R(-1,-1,0,0) + R(0,0,-1,0) + R(0,-1,1,-1) + R(0,-1,0,1), s1*s2*s3*s4*s1*s2*s3*s1: R(-1,-1,0,-1) + R(-1,-1,-1,1) + R(-2,0,0,0) + R(0,0,-1,-1) + R(0,0,-2,1) + R(0,-1,1,-2) + 4*R(0,-1,0,0) + R(0,-1,-1,2) + 2*R(-1,1,-1,0) + R(-1,0,1,-1) + R(-1,0,0,1) + R(1,0,-1,0) + R(1,-1,1,-1) + R(1,-1,0,1), s1*s2*s3*s4*s1*s2*s3*s1*s2: R(-1,-1,-1,0) + R(-2,0,0,-1) + R(-2,0,-1,1) + R(0,0,-2,0) + 3*R(0,-1,0,-1) + 3*R(0,-1,-1,1) + 2*R(-1,1,-1,-1) + 2*R(-1,1,-2,1) + R(-1,0,1,-2) + 5*R(-1,0,0,0) + R(-1,0,-1,2) + R(1,0,-1,-1) + R(1,0,-2,1) + R(1,-1,1,-2) + 3*R(1,-1,0,0) + R(1,-1,-1,2) + 2*R(0,1,-1,0) + R(0,0,1,-1) + R(0,0,0,1), s4*s1: 4*R(0,0,0,0) + R(-2,1,0,0) + R(-1,-1,1,0) + R(-1,0,-1,1) + 2*R(-1,0,0,-1) + R(1,-2,1,0) + R(1,-1,-1,1) + R(1,-1,0,-1) + R(-1,2,-1,0) + R(-1,1,1,-1) + R(0,1,-2,1) + R(0,1,-1,-1) + R(0,-1,2,-1) + R(0,0,1,-2), s4*s3*s1: 2*R(0,-1,0,-1) + R(0,-1,-1,1) + R(0,-2,1,0) + 2*R(-1,1,-1,-1) + R(-1,1,-2,1) + 2*R(-1,0,1,-2) + 5*R(-1,0,0,0) + R(-1,-1,2,-1) + R(-2,2,-1,0) + R(-2,1,1,-1) + R(1,0,-1,-1) + R(1,0,-2,1) + R(1,-1,1,-2) + 3*R(1,-1,0,0) + R(1,-2,2,-1) + R(0,1,0,-2) + 3*R(0,1,-1,0) + 3*R(0,0,1,-1) + R(-1,2,0,-1), s4*s3*s1*s2: R(0,0,-1,0) + R(0,-1,1,-1) + R(-1,1,0,-1), s3*s4*s3*s1: 3*R(-1,-1,0,-1) + 2*R(-1,-1,-1,1) + R(-1,-2,1,0) + 3*R(-2,1,-1,-1) + 2*R(-2,1,-2,1) + 2*R(-2,0,1,-2) + 6*R(-2,0,0,0) + R(-2,-1,2,-1) + R(-3,2,-1,0) + R(-3,1,1,-1) + 2*R(1,-2,0,-1) + R(1,-2,-1,1) + R(1,-3,1,0) + 9*R(0,0,-1,-1) + 6*R(0,0,-2,1) + 6*R(0,-1,1,-2) + 21*R(0,-1,0,0) + 3*R(0,-1,-1,2) + 3*R(0,-2,2,-1) + 2*R(0,-2,1,1) + 2*R(-1,2,-2,-1) + R(-1,2,-3,1) + 6*R(-1,1,0,-2) + 21*R(-1,1,-1,0) + 3*R(-1,1,-2,2) + 13*R(-1,0,1,-1) + 9*R(-1,0,0,1) + R(-1,-1,2,0) + R(-2,3,-2,0) + 3*R(-2,2,0,-1) + 2*R(-2,2,-1,1) + R(-2,1,1,0) + R(2,-1,-1,-1) + R(2,-1,-2,1) + R(2,-2,1,-2) + 3*R(2,-2,0,0) + R(2,-3,2,-1) + R(1,1,-2,-1) + R(1,1,-3,1) + 3*R(1,0,0,-2) + 13*R(1,0,-1,0) + 2*R(1,0,-2,2) + 9*R(1,-1,1,-1) + 6*R(1,-1,0,1) + R(1,-2,2,0) + R(0,2,-1,-2) + 3*R(0,2,-2,0) + 9*R(0,1,0,-1) + 6*R(0,1,-1,1) + 3*R(0,0,1,0) + R(-1,3,-1,-1) + R(-1,2,0,0), s4*s3*s1*s2*s1: R(0,0,-1,-1) + R(0,-1,1,-2) + 2*R(0,-1,0,0) + R(-1,1,0,-2) + 2*R(-1,1,-1,0) + 2*R(-1,0,1,-1) + R(1,0,-1,0) + R(1,-1,1,-1) + R(0,1,0,-1), s3*s4*s3*s1*s2: 2*R(-1,0,-1,0) + R(-1,-1,1,-1) + R(-2,1,0,-1) + R(1,-1,-1,0) + R(1,-2,1,-1) + R(0,1,-2,0) + 4*R(0,0,0,-1) + 2*R(0,0,-1,1) + R(0,-1,1,0) + R(-1,2,-1,-1) + R(-1,1,0,0), s3*s4*s3*s1*s2*s1: 2*R(-1,0,-1,-1) + R(-1,-1,1,-2) + 3*R(-1,-1,0,0) + R(-2,1,0,-2) + 3*R(-2,1,-1,0) + 2*R(-2,0,1,-1) + R(1,-1,-1,-1) + R(1,-2,1,-2) + 2*R(1,-2,0,0) + R(0,1,-2,-1) + 3*R(0,0,0,-2) + 9*R(0,0,-1,0) + 6*R(0,-1,1,-1) + 3*R(0,-1,0,1) + R(-1,2,-1,-2) + 2*R(-1,2,-2,0) + 6*R(-1,1,0,-1) + 3*R(-1,1,-1,1) + 2*R(-1,0,1,0) + R(2,-1,-1,0) + R(2,-2,1,-1) + R(1,1,-2,0) + 3*R(1,0,0,-1) + 2*R(1,0,-1,1) + R(1,-1,1,0) + R(0,2,-1,-1) + R(0,1,0,0), s3*s4*s1*s2*s3*s2: 4*R(0,0,0,0) + R(0,-2,1,-1) + R(0,-1,-1,0) + R(-2,1,0,0) + R(-2,2,-1,-1) + R(-1,1,-2,0) + R(-1,-1,1,0) + 2*R(-1,0,-1,1) + 4*R(-1,0,0,-1) + R(1,-2,1,0) + R(1,-1,-1,1) + 2*R(1,-1,0,-1) + R(-1,2,-1,0) + R(0,1,-2,1) + 2*R(0,1,-1,-1), s3*s4*s1*s2*s3*s2*s1: R(0,-1,0,-1) + R(-1,1,-1,-1) + 2*R(-1,0,0,0) + R(1,-1,0,0) + R(0,1,-1,0), s3*s4*s1*s2*s3*s1*s2*s1: R(0,-1,-1,-1) + R(0,-2,0,0) + R(-1,1,-2,-1) + 2*R(-1,0,0,-2) + 6*R(-1,0,-1,0) + 3*R(-1,-1,1,-1) + 2*R(-1,-1,0,1) + R(-2,2,-2,0) + 3*R(-2,1,0,-1) + 2*R(-2,1,-1,1) + R(1,-1,0,-2) + 3*R(1,-1,-1,0) + 2*R(1,-2,1,-1) + R(1,-2,0,1) + R(0,1,-1,-2) + 3*R(0,1,-2,0) + 9*R(0,0,0,-1) + 6*R(0,0,-1,1) + 3*R(0,-1,1,0) + 2*R(-1,2,-1,-1) + R(-1,2,-2,1) + 3*R(-1,1,0,0) + R(2,-1,0,-1) + R(2,-1,-1,1) + R(1,1,-1,-1) + R(1,1,-2,1) + 2*R(1,0,0,0), s4*s1*s2: 2*R(-1,0,-1,0) + 2*R(-1,-1,1,-1) + R(-1,-1,0,1) + 2*R(-2,1,0,-1) + R(-2,1,-1,1) + R(-2,0,1,0) + R(1,-1,-1,0) + R(1,-2,1,-1) + R(1,-2,0,1) + R(0,1,-2,0) + 5*R(0,0,0,-1) + 3*R(0,0,-1,1) + R(0,-1,2,-2) + 3*R(0,-1,1,0) + R(-1,2,-1,-1) + R(-1,2,-2,1) + R(-1,1,1,-2) + 3*R(-1,1,0,0) + R(-1,0,2,-1), s4*s1*s2*s3: R(0,-1,0,0) + R(-1,1,-1,0) + R(-1,0,1,-1), s4*s1*s2*s3*s2: 5*R(0,0,0,0) + R(0,-2,1,-1) + R(0,-1,-1,0) + 2*R(-2,1,0,0) + R(-2,2,-1,-1) + R(-2,1,1,-2) + R(-1,1,-2,0) + 2*R(-1,-1,1,0) + R(-1,-1,2,-2) + 2*R(-1,0,-1,1) + 5*R(-1,0,0,-1) + R(1,-2,1,0) + R(1,-1,-1,1) + 2*R(1,-1,0,-1) + R(-1,2,-1,0) + R(-1,1,1,-1) + R(0,1,-2,1) + 2*R(0,1,-1,-1) + R(0,-1,2,-1) + 2*R(0,0,1,-2), s1*s2*s3*s4*s3: R(-1,-1,0,0) + R(-2,1,-1,0) + R(-2,0,1,-1) + 2*R(0,0,-1,0) + 2*R(0,-1,1,-1) + R(0,-1,0,1) + 2*R(-1,1,0,-1) + R(-1,1,-1,1) + R(-1,0,1,0), s4*s1*s2*s3*s2*s1: R(0,-1,0,-1) + R(-1,1,-1,-1) + R(-1,0,1,-2) + 2*R(-1,0,0,0) + R(1,-1,0,0) + R(0,1,-1,0) + R(0,0,1,-1), s1*s2*s3*s4*s3*s2: R(-1,0,-1,0) + R(-1,-1,1,-1) + R(-2,1,0,-1) + 2*R(0,0,0,-1) + R(0,0,-1,1) + R(0,-1,1,0) + R(-1,1,0,0), s1*s2*s3*s4*s3*s2*s1: R(0,0,0,0) + R(-1,0,0,-1), s1*s2*s3*s4*s2*s3*s2: R(-1,-1,-1,0) + R(-1,-2,1,-1) + R(-2,1,-2,0) + 3*R(-2,0,0,-1) + 2*R(-2,0,-1,1) + R(-2,-1,1,0) + R(-3,2,-1,-1) + R(-3,1,0,0) + 2*R(0,0,-2,0) + 8*R(0,-1,0,-1) + 5*R(0,-1,-1,1) + R(0,-2,2,-2) + 3*R(0,-2,1,0) + 8*R(-1,1,-1,-1) + 5*R(-1,1,-2,1) + 5*R(-1,0,1,-2) + 16*R(-1,0,0,0) + 2*R(-1,0,-1,2) + 2*R(-1,-1,2,-1) + R(-1,-1,1,1) + R(-2,2,0,-2) + 3*R(-2,2,-1,0) + 2*R(-2,1,1,-1) + R(-2,1,0,1) + 4*R(1,0,-1,-1) + 2*R(1,0,-2,1) + 2*R(1,-1,1,-2) + 8*R(1,-1,0,0) + R(1,-1,-1,2) + R(1,-2,2,-1) + R(1,-2,1,1) + 2*R(0,1,0,-2) + 8*R(0,1,-1,0) + R(0,1,-2,2) + 5*R(0,0,1,-1) + 3*R(0,0,0,1) + R(-1,2,0,-1) + R(-1,2,-1,1), s1*s2*s3*s4*s2*s3*s2*s1: R(-1,-1,0,-1) + R(-2,1,-1,-1) + R(-2,0,0,0) + 2*R(0,0,-1,-1) + R(0,-1,1,-2) + 4*R(0,-1,0,0) + R(-1,1,0,-2) + 4*R(-1,1,-1,0) + 2*R(-1,0,1,-1) + R(-1,0,0,1) + 2*R(1,0,-1,0) + R(1,-1,1,-1) + R(1,-1,0,1) + R(0,1,0,-1) + R(0,1,-1,1), s1*s2*s3*s4*s2*s3*s1*s2*s1: R(-1,0,-1,-1) + R(-1,-1,0,0) + R(-2,1,-1,0) + R(0,0,0,-2) + 4*R(0,0,-1,0) + 2*R(0,-1,1,-1) + R(0,-1,0,1) + 2*R(-1,1,0,-1) + R(-1,1,-1,1) + R(1,0,0,-1) + R(1,0,-1,1), s4*s1*s2*s1: 3*R(-1,0,-1,-1) + 2*R(-1,0,-2,1) + 3*R(-1,-1,1,-2) + 9*R(-1,-1,0,0) + R(-1,-1,-1,2) + 2*R(-1,-2,2,-1) + R(-1,-2,1,1) + 2*R(-2,1,0,-2) + 6*R(-2,1,-1,0) + R(-2,1,-2,2) + 6*R(-2,0,1,-1) + 3*R(-2,0,0,1) + R(-2,-1,2,0) + 2*R(1,-1,-1,-1) + R(1,-1,-2,1) + 2*R(1,-2,1,-2) + 6*R(1,-2,0,0) + R(1,-2,-1,2) + R(1,-3,2,-1) + R(1,-3,1,1) + R(0,1,-2,-1) + R(0,1,-3,1) + 6*R(0,0,0,-2) + 21*R(0,0,-1,0) + 3*R(0,0,-2,2) + R(0,-1,2,-3) + 21*R(0,-1,1,-1) + 13*R(0,-1,0,1) + R(0,-2,3,-2) + 3*R(0,-2,2,0) + R(-1,2,-1,-2) + 3*R(-1,2,-2,0) + R(-1,2,-3,2) + R(-1,1,1,-3) + 13*R(-1,1,0,-1) + 9*R(-1,1,-1,1) + 3*R(-1,0,2,-2) + 9*R(-1,0,1,0) + R(-1,-1,3,-1) + 3*R(2,-1,-1,0) + 3*R(2,-2,1,-1) + 2*R(2,-2,0,1) + 2*R(1,1,-2,0) + 9*R(1,0,0,-1) + 6*R(1,0,-1,1) + 2*R(1,-1,2,-2) + 6*R(1,-1,1,0) + R(0,2,-1,-1) + R(0,2,-2,1) + R(0,1,1,-2) + 3*R(0,1,0,0) + R(0,0,2,-1), s4*s1*s2*s3*s1: 2*R(0,-1,0,-1) + R(0,-1,-1,1) + R(0,-2,1,0) + R(-1,1,-1,-1) + R(-1,1,-2,1) + R(-1,0,1,-2) + 4*R(-1,0,0,0) + R(-1,-1,2,-1) + 2*R(1,-1,0,0) + R(0,1,-1,0) + R(0,0,1,-1), s4*s1*s2*s3*s1*s2: 4*R(0,0,0,0) + R(0,-2,1,-1) + R(0,-1,-1,0) + R(-1,1,-2,0) + 2*R(-1,-1,1,0) + R(-1,-1,2,-2) + 2*R(-1,0,-1,1) + 4*R(-1,0,0,-1) + R(1,-2,1,0) + R(1,-1,-1,1) + 2*R(1,-1,0,-1) + R(0,1,-2,1) + R(0,1,-1,-1) + R(0,-1,2,-1) + R(0,0,1,-2), s1*s2*s3*s4*s3*s1: 2*R(-1,-1,0,-1) + R(-1,-1,-1,1) + R(-1,-2,1,0) + R(-2,1,-1,-1) + R(-2,1,-2,1) + R(-2,0,1,-2) + 3*R(-2,0,0,0) + R(-2,-1,2,-1) + 3*R(0,0,-1,-1) + 2*R(0,0,-2,1) + 3*R(0,-1,1,-2) + 9*R(0,-1,0,0) + R(0,-1,-1,2) + 2*R(0,-2,2,-1) + R(0,-2,1,1) + 2*R(-1,1,0,-2) + 6*R(-1,1,-1,0) + R(-1,1,-2,2) + 6*R(-1,0,1,-1) + 3*R(-1,0,0,1) + R(-1,-1,2,0) + 3*R(1,0,-1,0) + 3*R(1,-1,1,-1) + 2*R(1,-1,0,1) + 2*R(0,1,0,-1) + R(0,1,-1,1) + R(0,0,1,0), s4*s1*s2*s3*s1*s2*s1: R(0,-1,-1,-1) + R(0,-2,1,-2) + 2*R(0,-2,0,0) + R(-1,1,-2,-1) + 3*R(-1,0,0,-2) + 8*R(-1,0,-1,0) + R(-1,-1,2,-3) + 8*R(-1,-1,1,-1) + 4*R(-1,-1,0,1) + R(-2,2,-2,0) + 5*R(-2,1,0,-1) + 2*R(-2,1,-1,1) + R(-2,0,2,-2) + 2*R(-2,0,1,0) + 2*R(1,-1,0,-2) + 5*R(1,-1,-1,0) + 5*R(1,-2,1,-1) + 2*R(1,-2,0,1) + R(0,1,-1,-2) + 3*R(0,1,-2,0) + R(0,0,1,-3) + 16*R(0,0,0,-1) + 8*R(0,0,-1,1) + 3*R(0,-1,2,-2) + 8*R(0,-1,1,0) + 2*R(-1,2,-1,-1) + R(-1,2,-2,1) + 2*R(-1,1,1,-2) + 5*R(-1,1,0,0) + R(-1,0,2,-1) + 2*R(2,-1,0,-1) + R(2,-1,-1,1) + R(2,-2,1,0) + R(1,1,-1,-1) + R(1,1,-2,1) + R(1,0,1,-2) + 3*R(1,0,0,0) + R(1,-1,2,-1), s1*s2*s3*s4*s3*s1*s2: R(-1,0,-1,0) + R(-1,-1,1,-1) + 2*R(0,0,0,-1) + R(0,0,-1,1) + R(0,-1,1,0), s1*s2*s3*s4*s3*s1*s2*s1: R(-1,0,-1,-1) + R(-1,-1,1,-2) + 2*R(-1,-1,0,0) + R(-2,1,-1,0) + R(-2,0,1,-1) + R(0,0,0,-2) + 4*R(0,0,-1,0) + 4*R(0,-1,1,-1) + 2*R(0,-1,0,1) + 2*R(-1,1,0,-1) + R(-1,1,-1,1) + R(-1,0,1,0) + R(1,0,0,-1) + R(1,0,-1,1) + R(1,-1,1,0), s1*s2*s3*s4*s1*s2*s3*s2: R(-1,-1,-1,0) + R(-1,-2,1,-1) + 2*R(-2,0,0,-1) + R(-2,0,-1,1) + R(-2,-1,1,0) + R(0,0,-2,0) + 6*R(0,-1,0,-1) + 3*R(0,-1,-1,1) + R(0,-2,2,-2) + 3*R(0,-2,1,0) + 3*R(-1,1,-1,-1) + 2*R(-1,1,-2,1) + 3*R(-1,0,1,-2) + 9*R(-1,0,0,0) + R(-1,0,-1,2) + 2*R(-1,-1,2,-1) + R(-1,-1,1,1) + 2*R(1,0,-1,-1) + R(1,0,-2,1) + 2*R(1,-1,1,-2) + 6*R(1,-1,0,0) + R(1,-1,-1,2) + R(1,-2,2,-1) + R(1,-2,1,1) + 3*R(0,1,-1,0) + 3*R(0,0,1,-1) + 2*R(0,0,0,1), s1*s2*s3*s4*s1*s2*s3*s2*s1: R(-1,-1,0,-1) + R(-2,0,0,0) + R(0,0,-1,-1) + R(0,-1,1,-2) + 4*R(0,-1,0,0) + 2*R(-1,1,-1,0) + 2*R(-1,0,1,-1) + R(-1,0,0,1) + R(1,0,-1,0) + R(1,-1,1,-1) + R(1,-1,0,1), s1*s2*s3*s4*s1*s2*s3*s1*s2*s1: 21*R(0,0,0,0) + R(-1,-2,0,0) + R(-1,-1,-1,-1) + R(-3,1,-1,1) + R(-3,1,0,-1) + R(-2,-1,0,1) + 2*R(-2,-1,1,-1) + 3*R(-2,0,-1,0) + R(-2,0,0,-2) + R(0,0,-2,-1) + 3*R(0,-2,0,1) + 6*R(0,-2,1,-1) + 9*R(0,-1,-1,0) + 3*R(0,-1,0,-2) + 6*R(-2,1,0,0) + 2*R(-2,2,-2,1) + 3*R(-2,2,-1,-1) + 2*R(-2,1,1,-2) + R(-2,1,-1,2) + 6*R(-1,1,-2,0) + 2*R(-1,1,-1,-2) + 9*R(-1,-1,1,0) + 3*R(-1,-1,2,-2) + R(-1,-1,0,2) + R(-1,0,1,-3) + 13*R(-1,0,-1,1) + 21*R(-1,0,0,-1) + 3*R(2,-1,0,0) + R(2,0,-2,1) + R(2,0,-1,-1) + R(2,-1,1,-2) + R(2,-1,-1,2) + 2*R(1,1,-1,0) + R(1,0,1,-1) + R(1,0,0,1) + 3*R(1,0,-2,0) + R(1,0,-1,-2) + 6*R(1,-2,1,0) + 2*R(1,-2,2,-2) + R(1,-2,0,2) + R(1,-1,1,-3) + 9*R(1,-1,-1,1) + 13*R(1,-1,0,-1) + 3*R(-1,2,-1,0) + 2*R(-1,1,1,-1) + R(-1,1,0,1) + 6*R(0,1,-2,1) + 9*R(0,1,-1,-1) + 3*R(0,-1,2,-1) + 2*R(0,-1,1,1) + 6*R(0,0,1,-2) + 3*R(0,0,-1,2)}

In [12]:
for w in W:
    print(str("w = ") + str(w))
    print(str("f^w: ") + str(d_sup[list(W).index(w)]))
    print(str("g^w: ") + str(newkl[w]))
    print()

w = 1
f^w: 120*a4(0,0,0,0)
g^w: a4(0,0,0,0)

w = s3
f^w: 24*a4(0,-1,0,0) + 24*a4(-1,1,-1,0) + 12*a4(-1,0,1,-1) + 12*a4(-1,0,0,1) + 24*a4(1,0,-1,0) + 12*a4(1,-1,1,-1) + 12*a4(1,-1,0,1) + 12*a4(0,1,0,-1) + 12*a4(0,1,-1,1)
g^w: 2*a4(0,-1,0,0) + 2*a4(-1,1,-1,0) + a4(-1,0,1,-1) + a4(-1,0,0,1) + 2*a4(1,0,-1,0) + a4(1,-1,1,-1) + a4(1,-1,0,1) + a4(0,1,0,-1) + a4(0,1,-1,1)

w = s3*s2
f^w: 24*a4(0,0,-1,0) + 12*a4(0,-1,1,-1) + 12*a4(0,-1,0,1) + 12*a4(-1,1,0,-1) + 12*a4(-1,1,-1,1) + 12*a4(1,0,0,-1) + 12*a4(1,0,-1,1)
g^w: 2*a4(0,0,-1,0) + a4(0,-1,1,-1) + a4(0,-1,0,1) + a4(-1,1,0,-1) + a4(-1,1,-1,1) + a4(1,0,0,-1) + a4(1,0,-1,1)

w = s3*s4
f^w: 24*a4(-1,0,0,0) + 24*a4(1,-1,0,0) + 24*a4(0,1,-1,0)
g^w: a4(-1,0,0,0) + a4(1,-1,0,0) + a4(0,1,-1,0)

w = s3*s2*s1
f^w: 24*a4(0,0,0,-1) + 24*a4(0,0,-1,1)
g^w: a4(0,0,0,-1) + a4(0,0,-1,1)

w = s3*s4*s2
f^w: 8*a4(-1,0,-1,0) + 4*a4(-1,-1,1,-1) + 4*a4(-1,-1,0,1) + 4*a4(-2,1,0,-1) + 4*a4(-2,1,-1,1) + 8*a4(1,-1,-1,0) + 4*a4(1,-2,1,-1) + 4*a4(1,-2,0,1) + 8*a4(0,1,-2,

In [37]:
print(list(W))

[1, s3*s1, s2*s3*s1*s2, s1*s2*s3*s1*s2*s1, s1*s2, s1*s2*s3*s1, s3*s2, s2*s3*s2*s1, s2*s1, s2*s3, s1*s2*s3*s2, s3*s1*s2*s1, s1, s3, s1*s2*s3*s1*s2, s2*s3*s1*s2*s1, s2, s2*s3*s1, s3*s1*s2, s1*s2*s3*s2*s1, s1*s2*s1, s1*s2*s3, s2*s3*s2, s3*s2*s1]


In [40]:
trouble = []
for w in W:
    print("---")
    print(w)
    print("---")
    for y in W:
        pairingvalue = woke_pairing(newkl[w], d_dual[list(W).index(y)])
        if pairingvalue != 0 and w != y:
            if w not in trouble:
                trouble.append(w)
            print(y)
            print(pairingvalue)
            print()
print(trouble)

---
1
---
---
s3*s1
---
1
1/8*A3(0,0,0)

---
s2*s3*s1*s2
---
---
s1*s2*s3*s1*s2*s1
---
s1*s2
-1/4*A3(0,0,0)

s3*s2
-1/4*A3(0,0,0)

s2
1/4*A3(0,0,0)

s3*s1*s2
1/4*A3(0,0,0)

s1*s2*s3
1/6*A3(0,0,1)

s3*s2*s1
1/6*A3(1,0,0)

---
s1*s2
---
---
s1*s2*s3*s1
---
1
1/24*A3(0,0,0)

---
s3*s2
---
---
s2*s3*s2*s1
---
1
1/24*A3(0,0,0)

---
s2*s1
---
---
s2*s3
---
---
s1*s2*s3*s2
---
s1
1/6*A3(0,0,0)

s3*s2*s1
1/6*A3(0,0,0)

---
s3*s1*s2*s1
---
s3
1/6*A3(0,0,0)

s1*s2*s3
1/6*A3(0,0,0)

---
s1
---
---
s3
---
---
s1*s2*s3*s1*s2
---
s2*s1
1/6*A3(0,0,0)

---
s2*s3*s1*s2*s1
---
s2*s3
1/6*A3(0,0,0)

---
s2
---
---
s2*s3*s1
---
1
1/12*A3(0,0,0)

---
s3*s1*s2
---
---
s1*s2*s3*s2*s1
---
1
1/24*A3(0,0,0)

---
s1*s2*s1
---
1
1/24*A3(0,0,1)

s2*s3
1/6*A3(0,0,0)

s1*s2*s3
1/6*A3(0,0,0)

---
s1*s2*s3
---
---
s2*s3*s2
---
1
1/24*A3(1,0,0)

s2*s1
1/6*A3(0,0,0)

s3*s2*s1
1/6*A3(0,0,0)

---
s3*s2*s1
---
[s3*s1, s1*s2*s3*s1*s2*s1, s1*s2*s3*s1, s2*s3*s2*s1, s1*s2*s3*s2, s3*s1*s2*s1, s1*s2*s3*s1*s2, s2*s3*s1*s2*s1, s2*s

In [10]:
trouble = list(W)

for w in trouble:
    print(w0*w)
    print("---")
    for y in W:
        pairingvalue = woke_pairing(newkl[w], d_dual[list(W).index(y)])
        if pairingvalue != 0 and w != y:
            print(w0*y)
            print(f"w1 ≤_L w2: {left_cell_leq(w0*y, w0*w)}")
            #print(pairingvalue)
    print()
    print("---")

s1*s2*s3*s4*s1*s2*s3*s1*s2*s1
---

---
s1*s2*s3*s4*s2*s3*s1*s2*s1
---

---
s1*s2*s3*s4*s2*s3*s2*s1
---

---
s3*s4*s1*s2*s3*s1*s2*s1
---

---
s1*s2*s3*s4*s2*s3*s2
---

---
s3*s4*s1*s2*s3*s2*s1
---
s1*s2*s3*s4*s1*s2*s3*s1*s2*s1
w1 ≤_L w2: True
s1*s2*s3*s4*s2*s3*s2
w1 ≤_L w2: True

---
s3*s4*s1*s2*s3*s2
---
s1*s2*s3*s4*s1*s2*s3*s1*s2*s1
w1 ≤_L w2: True

---
s3*s4*s3*s1*s2*s1
---

---
s3*s4*s3*s1*s2
---
s3*s4*s1*s2*s3*s1*s2*s1
w1 ≤_L w2: True

---
s3*s4*s3*s1
---

---
s1*s2*s3*s4*s1*s2*s3*s2*s1
---

---
s1*s2*s3*s4*s3*s1*s2*s1
---

---
s1*s2*s3*s4*s3*s2*s1
---
s1*s2*s3*s4*s1*s2*s3*s1*s2*s1
w1 ≤_L w2: True
s3*s4*s1*s2*s3*s1*s2*s1
w1 ≤_L w2: True
s1*s2*s3*s4*s2*s3*s2
w1 ≤_L w2: True
s3*s4*s1*s2*s3*s2
w1 ≤_L w2: True
s4*s1*s2*s3*s1*s2*s1
w1 ≤_L w2: True
s4*s1*s2*s3*s2
w1 ≤_L w2: True
s1*s2*s3*s4*s1*s2*s3*s2
w1 ≤_L w2: True
s4*s1*s2*s3*s1*s2
w1 ≤_L w2: True

---
s4*s1*s2*s3*s1*s2*s1
---

---
s1*s2*s3*s4*s3*s2
---
s1*s2*s3*s4*s1*s2*s3*s1*s2*s1
w1 ≤_L w2: True
s3*s4*s1*s2*s3*s1*s2*s1
w1 ≤_L w2: 

In [11]:
for w in W:
    for y in W:
        print(f"w1 ≤_L w2: {left_cell_leq(w0*y, w0*w)}")


w1 ≤_L w2: True
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: False
w1 ≤_L w2: Fals

In [9]:
def nqpair(w, y):
    #new/normalized q_pairing
    return woke_pairing(d_qsub[list(W).index(w)], d_dual[list(W).index(y)])

table = []

Mw = {}

for w in invols:
    table.append([str(w), str(nqpair(w0*w, w0*w))])

for row in table:
    print('| {:20} | {:1}'.format(*row))

| 1                    | A3(0,0,0)
| s3*s1                | -A3(9,0,9) + A3(10,0,10)
| s2*s3*s1*s2          | A3(0,8,0) + A3(0,10,0)
| s1*s2*s3*s1*s2*s1    | A3(10,10,10)
| s1                   | A3(10,0,0)
| s3                   | A3(0,0,10)
| s2                   | -A3(0,8,0) + A3(0,10,0)
| s1*s2*s3*s2*s1       | A3(9,0,9) + A3(10,0,10)
| s1*s2*s1             | A3(10,10,0)
| s2*s3*s2             | A3(0,10,10)


In [10]:
l = [[s1*s2*s3*s1, s2*s3*s2*s1, s1*s2*s3*s2, s3*s1*s2*s1, s1*s2*s3*s1*s2, s2*s3*s1*s2*s1, s1*s2*s3*s2*s1, s1*s2*s1, s2*s3*s2], [s1*s2, s3*s2, s2*s1, s2*s3, s1, s3, s2, s1*s2*s3, s3*s2*s1], [s1*s2*s3*s1*s2*s1], [s3*s1, s2*s3*s1*s2, s2*s3*s1, s3*s1*s2], [s1*s1]]
bigs = 0
for a in l:
    s = 0
    for b in a:
        if b.length() < (b*s1).length():
            s += nqpair(b, b)
    bigs += s
    print(s)
print(bigs)

3*A3(10,0,0)
3*A3(9,0,9) + 3*A3(10,0,10) + 3*A3(10,10,0)
0
-2*A3(9,0,9) + 2*A3(10,0,10)
A3(10,10,10)
3*A3(10,0,0) + A3(9,0,9) + 5*A3(10,0,10) + 3*A3(10,10,0) + A3(10,10,10)


In [11]:
final_dual = {}
for i in range(len(W)):
    final_dual[W[i]] = d_dual[i]
print(final_dual)

{1: a3(0,0,0), s3*s1: 3/4*a3(0,0,0) + 1/4*a3(-2,1,0) + 1/4*a3(-1,-1,1) + 1/2*a3(-1,0,-1) + 1/4*a3(1,-2,1) + 1/4*a3(1,-1,-1) + 1/4*a3(-1,2,-1) + 1/4*a3(0,1,-2), s2*s3*s1*s2: 1/2*a3(0,-1,0), s1*s2*s3*s1*s2*s1: 1/24*a3(-1,-1,-1) + 1/12*a3(0,-1,0) - 1/24*a3(-1,1,-1), s1*s2: 1/2*a3(0,-1,0) + 1/2*a3(-1,1,-1) + 1/2*a3(-1,0,1), s1*s2*s3*s1: 1/4*a3(-1,-1,1) + 1/4*a3(-1,0,-1), s3*s2: 1/2*a3(0,-1,0) + 1/2*a3(-1,1,-1) + 1/2*a3(1,0,-1), s2*s3*s2*s1: 1/4*a3(-1,0,-1) + 1/4*a3(1,-1,-1), s2*s1: 1/2*a3(0,0,-1) + 1/2*a3(0,-1,1), s2*s3: 1/2*a3(-1,0,0) + 1/2*a3(1,-1,0), s1*s2*s3*s2: 1/6*a3(-1,-1,0) + 1/6*a3(-2,1,-1), s3*s1*s2*s1: 1/6*a3(0,-1,-1) + 1/6*a3(-1,1,-2), s1: -1/2*a3(0,0,-1) - 1/2*a3(0,-1,1) - 1/2*a3(-1,1,0), s3: -1/2*a3(-1,0,0) - 1/2*a3(1,-1,0) - 1/2*a3(0,1,-1), s1*s2*s3*s1*s2: -1/6*a3(-1,-1,0), s2*s3*s1*s2*s1: -1/6*a3(0,-1,-1), s2: -a3(0,-1,0) - 1/2*a3(-1,1,-1) - 1/2*a3(-1,0,1) - 1/2*a3(1,0,-1) - 1/2*a3(1,-1,1), s2*s3*s1: -1/4*a3(0,0,0) - 1/4*a3(-1,-1,1) - 1/4*a3(-1,0,-1) - 1/4*a3(1,-2,1) - 1/4*

In [9]:
from sage.combinat.rsk import RSK

T = CartanType(['A', 4])
W = WeylGroup(T, prefix="s")
[s1, s2, s3, s4] = W.simple_reflections()

def left_cell_shape(w):
    """
    Get the shape of the left tableau (Q tableau) for a Weyl group element.
    For type A_n, convert permutation to RSK.
    """
    # Convert Weyl group element to permutation (in one-line notation)
    perm = w.to_permutation()
    P, Q = RSK(perm)
    return Q.shape()  # Left cell corresponds to Q tableau shape

def left_cells_comparable(w1, w2):
    """
    Check if w1 and w2 are comparable in the left cell order.
    This means their partition shapes are comparable in dominance order.
    """
    shape1 = left_cell_shape(w1)
    shape2 = left_cell_shape(w2)
    
    # Check if shape1 dominates shape2 or vice versa
    return shape1.dominates(shape2) or shape2.dominates(shape1)

def same_left_cell(w1, w2):
    """Check if w1 and w2 are in the same left cell."""
    return left_cell_shape(w1) == left_cell_shape(w2)

def left_cell_leq(w1, w2):
    """
    Check if w1 ≤_L w2 in left cell order.
    This means shape(w2) dominates shape(w1).
    """
    shape1 = left_cell_shape(w1)
    shape2 = left_cell_shape(w2)
    return shape2.dominates(shape1)

# Example:
w1 = s1*s2*s1*s3*s2*s1*s4
w2 = s1 * s2 * s1

print(f"Shape of w1: {left_cell_shape(w1)}")
print(f"Shape of w2: {left_cell_shape(w2)}")
print(f"w1 ≤_L w2: {left_cell_leq(w1, w2)}")
print(f"Same left cell: {same_left_cell(w1, w2)}")

Shape of w1: [2, 1, 1, 1]
Shape of w2: [3, 1, 1]
w1 ≤_L w2: True
Same left cell: False
